# 02 - Scanned PDF NER

Pipeline: PDF -> PyMuPDF -> GLiNER -> predicted entities -> strict/relaxed evaluation.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import load_gliner_model, predict_dataset
from ner.pdf import extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl


C:\Users\Alexis\Documents\MODULS POBLENOU\2do CURSO\INTELIGENCIA ARTIFICIAL\ACTIVITATS\ner_project\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\Alexis\Documents\MODULS POBLENOU\2do CURSO\INTELIGENCIA ARTIFICIAL\ACTIVITATS\ner_project\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_scanned'
OUTPUT_PATH = '../results/pdf_scanned/gliner_large_validation.jsonl'

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]


In [4]:
gold_samples = load_jsonl(VALIDATION_PATH)
pdf_samples = extract_pdf_dataset(PDF_DIR, gold_samples)
model = load_gliner_model("gliner-community/gliner_small-v2.5")
responses = predict_dataset(model, pdf_samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)


C:\Users\Alexis\Documents\MODULS POBLENOU\2do CURSO\INTELIGENCIA ARTIFICIAL\ACTIVITATS\ner_project\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [5]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH, True)
strict_results['micro']
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH, True)
relaxed_results['occurrences']['micro']



STRICT EVALUATION
Micro -> P: 0.76 | R: 0.86 | F1: 0.81 | TP: 1773 | FP: 545 | FN: 277

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00     500       0       0
ID_CARD_NUMBER              0.92   1.00   0.96     200      17       0
ADDRESS                     0.00   0.01   0.00       1     415     199
REGISTRATION_NUMBER         1.00   0.94   0.97     236       1      14
DATE                        1.00   1.00   1.00     399       0       1
AMOUNT                      0.95   0.92   0.94     184       9      16
ORGANIZATION                0.71   0.84   0.77     253     103      47

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.76 | R: 0.86 | F1: 0.81 | TP: 1773 | FP: 545 | FN: 277

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------------------------------------------------

{'precision': 0.7648835202761001,
 'recall': 0.8648780487804878,
 'f1': 0.8118131868131867,
 'true_positives': 1773,
 'false_positives': 545,
 'false_negatives': 277}